<div>

  <b>Escuela Politécnica Nacional</b><br>
  <small>Facultad de Ingeniería de Sistemas</small><br>
  <small>Ingeniería en Ciencias de la Computación</small>

  <hr>

  <div style="display:flex; justify-content:space-between;">
    <div>
      Estudiante: <b>Mateo Cumbal</b><br>
      Fecha: <b>2026-06-17</b>
    </div>
    <div style="text-align:right;">
      Asignatura: <b>Recuperación de la Información</b><br>
      Paralelo: <b>GR1CC</b>
    </div>
  </div>

  <hr>

  <div style="text-align:center;">
    <h1><b>Ejercicio 10 — Re-ranking</b></h1>
  </div>

</div>

In [9]:
# !pip install beir pandas sentence-transformers pytrec_eval rank_bm25 scikit-learn

In [2]:
import numpy as np
import pandas as pd
import pytrec_eval
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

c:\Users\Acer123\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\beir\util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Parte 1 — Carga del corpus

In [3]:
DATASET    = "scifact"
DIRECTORIO = "/content/data/beir"
url        = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET}.zip"

util.download_and_unzip(url, DIRECTORIO)

/content/data/beir\scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'/content/data/beir\\scifact'

In [4]:
ruta = f"{DIRECTORIO}/{DATASET}"
corpus, consultas, qrels = GenericDataLoader(ruta).load(split="test")

df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_consultas = (
    pd.DataFrame.from_dict(consultas, orient="index", columns=["consulta"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_qrels = pd.DataFrame(
    [{"query_id": qid, "doc_id": did, "relevancia": rel}
     for qid, docs in qrels.items()
     for did, rel in docs.items()]
)

print(f"Corpus: {len(df_corpus)} docs | Consultas: {len(df_consultas)} | Qrels: {len(df_qrels)} pares")

  0%|          | 0/5183 [00:00<?, ?it/s]

Corpus: 5183 docs | Consultas: 300 | Qrels: 339 pares


In [13]:
# Consulta de ejemplo con documentos relevantes
qid_ejemplo = "133"
print("Consulta:", df_consultas.loc[df_consultas["query_id"] == qid_ejemplo, "consulta"].values[0])
print("\nDocs relevantes:")
df_qrels[(df_qrels["query_id"] == qid_ejemplo) & (df_qrels["relevancia"] > 0)]

Consulta: Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Docs relevantes:


,query_id,doc_id,relevancia
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2 — Retrieval inicial con BM25 (baseline)

In [14]:
corpus_tokenizado = [str(t).lower().split() for t in df_corpus["text"]]
bm25 = BM25Okapi(corpus_tokenizado)
print(f"Índice BM25 construido sobre {len(corpus_tokenizado)} documentos")

Índice BM25 construido sobre 5183 documentos


In [15]:
TOP_K    = 100
run_bm25 = {}

for _, fila in df_consultas.iterrows():
    qid      = str(fila["query_id"])
    tokens_q = str(fila["consulta"]).lower().split()
    scores   = bm25.get_scores(tokens_q)
    indices  = np.argsort(scores)[::-1][:TOP_K]
    run_bm25[qid] = {str(df_corpus.iloc[i]["doc_id"]): float(scores[i]) for i in indices}

print(f"Retrieval completado para {len(run_bm25)} consultas")

Retrieval completado para 300 consultas


In [16]:
# Qrels en formato pytrec_eval
qrels_str = {
    str(qid): {str(did): int(rel) for did, rel in docs.items()}
    for qid, docs in qrels.items()
}

evaluador     = pytrec_eval.RelevanceEvaluator(qrels_str, {"ndcg_cut_10", "recall_10"})
metricas_bm25 = evaluador.evaluate(run_bm25)

ndcg_bm25   = np.mean([m["ndcg_cut_10"] for m in metricas_bm25.values()])
recall_bm25 = np.mean([m["recall_10"]   for m in metricas_bm25.values()])

print(f"BM25 — nDCG@10: {ndcg_bm25:.4f} | Recall@10: {recall_bm25:.4f}")

BM25 — nDCG@10: 0.5438 | Recall@10: 0.6688


## Parte 3 — Re-ranking con Cross-Encoder

In [ ]:
modelo_ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
textos    = dict(zip(df_corpus["doc_id"].astype(str), df_corpus["text"]))

run_ce = {}
for _, fila in df_consultas.iterrows():
    qid      = str(fila["query_id"])
    if qid not in run_bm25:
        continue
    consulta   = str(fila["consulta"])
    candidatos = list(run_bm25[qid].keys())
    pares      = [(consulta, textos.get(did, "")) for did in candidatos]
    scores     = modelo_ce.predict(pares)
    run_ce[qid] = {did: float(s) for did, s in zip(candidatos, scores)}

print(f"Re-ranking CE completado para {len(run_ce)} consultas")

## Parte 4 — Re-ranking con LTR (Random Forest)

In [18]:
# 1. Feature engineering
filas_ltr = [
    {
        "query_id":   qid,
        "doc_id":     doc_id,
        "score_bm25": score_bm25,
        "score_ce":   run_ce.get(qid, {}).get(doc_id, 0.0),
        "longitud":   len(textos.get(doc_id, "").split()),
        "etiqueta":   qrels_str.get(qid, {}).get(doc_id, 0),
    }
    for qid in run_bm25
    for doc_id, score_bm25 in run_bm25[qid].items()
]

df_ltr = pd.DataFrame(filas_ltr)
print(f"Dataset LTR: {len(df_ltr)} filas")

# 2. Split por query_id para evitar data leakage
ids_train, ids_test = train_test_split(
    df_ltr["query_id"].unique(), test_size=0.3, random_state=42
)
tren   = df_ltr[df_ltr["query_id"].isin(ids_train)]
prueba = df_ltr[df_ltr["query_id"].isin(ids_test)].copy()

# 3. Entrenamiento
caracteristicas = ["score_bm25", "score_ce", "longitud"]
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
modelo_rf.fit(tren[caracteristicas], tren["etiqueta"])

# 4. Inferencia y construcción del run
prueba["score_ltr"] = modelo_rf.predict(prueba[caracteristicas])
run_ltr = {
    qid: dict(zip(
        prueba.loc[prueba["query_id"] == qid, "doc_id"],
        prueba.loc[prueba["query_id"] == qid, "score_ltr"]
    ))
    for qid in ids_test
}

print(f"LTR completado — {len(run_ltr)} consultas en test")

Dataset LTR: 30000 filas
LTR completado — 90 consultas en test


## Parte 5 — Evaluación comparativa (nDCG@10, MAP, Recall@10)

In [19]:
# Solo evaluamos sobre las queries del conjunto de test
qrels_test      = {qid: qrels_str[qid] for qid in ids_test if qid in qrels_str}
evaluador_final = pytrec_eval.RelevanceEvaluator(qrels_test, {"ndcg_cut_10", "recall_10", "map"})

run_bm25_test = {qid: run_bm25[qid] for qid in ids_test if qid in run_bm25}
run_ce_test   = {qid: run_ce[qid]   for qid in ids_test if qid in run_ce}

def promediar(metricas: dict) -> dict:
    """Extrae el promedio de nDCG@10, MAP y Recall@10."""
    return {
        "nDCG@10":   np.mean([m.get("ndcg_cut_10", 0) for m in metricas.values()]),
        "MAP":       np.mean([m.get("map",         0) for m in metricas.values()]),
        "Recall@10": np.mean([m.get("recall_10",   0) for m in metricas.values()]),
    }

resultados = {
    "BM25 (baseline)":     promediar(evaluador_final.evaluate(run_bm25_test)),
    "Cross-Encoder":       promediar(evaluador_final.evaluate(run_ce_test)),
    "LTR (Random Forest)": promediar(evaluador_final.evaluate(run_ltr)),
}

df_resultados = pd.DataFrame.from_dict(resultados, orient="index")
print(df_resultados.round(4))

                     nDCG@10     MAP  Recall@10
BM25 (baseline)       0.6507  0.6107     0.7704
Cross-Encoder         0.7216  0.6942     0.7963
LTR (Random Forest)   0.7179  0.6915     0.7907
